#Import

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

#Read Bronze Layer

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

#Silver Transformations

In [0]:
df.limit(10).display()

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df.withColumn("field",trim(col("field")))


##Customer ID Cleanup


In [0]:
# Remove the "NAS" prefix from customer IDs when present.
df = df.withColumn(
    "cid",
    when(col("cid").startswith("NAS"),
           substring(col("cid"), 4, length(col("cid"))))
     .otherwise(col("cid"))
)

##Birthdate Validation

In [0]:
df = df.withColumn(
    "bdate",
    when(col("bdate")>current_date(),None)
    .otherwise(col("bdate"))
)

##Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    when(upper(col("gen")).isin("M","MALE"),"Male")
    .when(upper(col("gen")).isin("F","FEMALE"),"Female")
    .otherwise("n/a")
)

##Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}

for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

##Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customers")

#Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers